In [7]:
# Chromeブラウザを利用して、スクレイピングをするため、Chromeブラウザをインストール
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' | tee /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install google-chrome-stable

OK
deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,218 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,977 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [61.6 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:13 http://security.ubu

In [2]:
# SeleniumとChromeDriverを自動で管理するためのインストール
!pip install selenium
!pip install webdriver-manager

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 17.3 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [3]:
!pip install beautifulsoup4

In [9]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import pandas as pd
import time

In [10]:
#  Chromeブラウザ起動オプションを指定
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

In [11]:
# ChromeDriverManagerを使用してChromeDriverを自動的に管理
service = Service(ChromeDriverManager().install())

# インスタンスを作成し、変数chrome_driverに格納
chrome_driver = webdriver.Chrome(service=service, options=chrome_options)

In [12]:
# 日経新聞のWebサイトを開く
chrome_driver.get('https://www.nikkei.com/markets/worldidx/chart/nk225/?type=6month')

In [13]:
# HTML解析（BeautifulSoup）
def extract_stock_data(html):
  soup = BeautifulSoup(html, 'html.parser')
  try:
    date = soup.select_one(".highcharts-tooltip td:nth-child(1)").text.strip()
    open_price = soup.select_one(".highcharts-tooltip td:nth-child(2)").text.strip()
    high_price = soup.select_one(".highcharts-tooltip td:nth-child(3)").text.strip()
    low_price = soup.select_one(".highcharts-tooltip td:nth-child(4)").text.strip()
    close_price = soup.select_one(".highcharts-tooltip td:nth-child(5)").text.strip()

    return [date, open_price, high_price, low_price, close_price]
  except:
    return None

In [14]:
def get_stock_values(driver, url):
    from selenium.webdriver.common.by import By
    from selenium.webdriver.common.action_chains import ActionChains
    import time
    from datetime import datetime, timedelta

    driver.get(url)
    time.sleep(3)

    actions = ActionChains(driver)

    graph = driver.find_element(By.CLASS_NAME, "highcharts-container")
    width = graph.size["width"]

    actions.move_to_element(graph).perform()

    results = []
    seen_dates = set()  # ←重複防止用

    # 今日から1ヶ月前の日付
    one_month_ago = datetime.today() - timedelta(days=30)

    for i in range(width):
      x_offset = width // 2 - i
      actions.move_to_element_with_offset(graph, x_offset, 0).perform()
      time.sleep(0.01)

      html = driver.page_source
      data = extract_stock_data(html)

      if data:
          date_str = data[0]

          # 日付フォーマット
          date_str = date_str.replace("/", "-")

          try:
              date_obj = datetime.strptime(date_str, "%Y-%m-%d")
              date_str = date_obj.strftime("%Y-%m-%d")
          except:
              continue

          # ✔ 重複チェック＆追加
          if date_str not in seen_dates:
              seen_dates.add(date_str)
              data[0] = date_str
              results.append(data)

          # ✔ 最後にbreak判定（重要）
          if date_obj < one_month_ago:
              break

    return results


# メイン処理
if __name__ == '__main__':
  # ドライバ準備
  service = Service(ChromeDriverManager().install())
  # Chromeオプションを適用してドライバーを作成
  driver = webdriver.Chrome(service=service, options=chrome_options)

  url = 'https://www.nikkei.com/markets/worldidx/chart/nk225/?type=6month'

  # 開始時間
  start_time = time.time()

  # データ取得
  stock_data_list = get_stock_values(driver,url)

  # 終了時間
  end_time = time.time()

  # 実行時間
  print(f'スクレイピング時間：{end_time - start_time:.2f}秒')

  # 結果表示
  if stock_data_list:
      for data in stock_data_list:
        print(f'{data[0]}, {data[1]}, {data[2]}, {data[3]}, {data[4]}')
  else:
      print("No stock data was retrieved.")

  driver.quit()

スクレイピング時間：101.23秒
2026-04-22, 始値: 59,104.11, 高値: 59,708.21, 安値: 59,005.48, 終値: 59,585.86
2026-04-21, 始値: 59,031.51, 高値: 59,611.91, 安値: 59,004.76, 終値: 59,349.17
2026-04-20, 始値: 58,821.16, 高値: 59,169.13, 安値: 58,687.96, 終値: 58,824.89
2026-04-17, 始値: 59,255.09, 高値: 59,381.25, 安値: 58,475.9, 終値: 58,475.9
2026-04-16, 始値: 58,479.83, 高値: 59,688.1, 安値: 58,428.19, 終値: 59,518.34
2026-04-15, 始値: 58,265.18, 高値: 58,585.95, 安値: 58,028.75, 終値: 58,134.24
2026-04-14, 始値: 57,085.65, 高値: 57,979.82, 安値: 57,010.18, 終値: 57,877.39
2026-04-13, 始値: 56,421.46, 高値: 56,765.72, 安値: 56,232.78, 終値: 56,502.77
2026-04-10, 始値: 56,265.77, 高値: 57,012.77, 安値: 56,251.18, 終値: 56,924.11
2026-04-09, 始値: 56,199.86, 高値: 56,406.49, 安値: 55,763.05, 終値: 55,895.32
2026-04-08, 始値: 54,386.65, 高値: 56,424.63, 安値: 54,380.02, 終値: 56,308.42
2026-04-07, 始値: 53,571.28, 高値: 53,916.35, 安値: 53,156.94, 終値: 53,429.56
2026-04-06, 始値: 53,205.93, 高値: 54,039.34, 安値: 53,205.93, 終値: 53,413.68
2026-04-03, 始値: 53,039.4, 高値: 53,426.31, 安値: 52,925.1, 終値: 53,